# Solve an existing problem with an existing algorithm

**Scenario:** the problem you want to solve is already implemented in
`qrisp_optimization.problems`, and so is the algorithm you want to use.
All you have to do is plug the two together.

In this guide you will

1. instantiate a [`MaxCutProblem`][qrisp_optimization.problems.MaxCutProblem],
2. compute its exact solutions with [`Bruteforce`][qrisp_optimization.Bruteforce] as a reference,
3. solve it with [`QAOA`][qrisp_optimization.QAOA], including a custom optimization metric,
4. solve it with the parameter-free [`LRQAOA`][qrisp_optimization.LRQAOA],
5. swap in a different problem without touching the solver code.

## Which problem works with which algorithm?

Algorithms never depend on concrete problem classes, only on a *requirements interface*.
A problem can be handed to every algorithm whose interface it implements; the
[overview on the home page](../index.md#which-problem-works-with-which-algorithm) lists
which problems and algorithms fit together.

You can also check this in code: a problem satisfies an interface exactly when it is an
instance of it.

In [1]:
import networkx as nx
import numpy as np
from qrisp import QuantumArray

from qrisp_optimization import (
    LRQAOA,
    QAOA,
    Bruteforce,
    BruteforceRequirements,
    QAOARequirements,
)
from qrisp_optimization.problems import MaxCutProblem, QUBOProblem
from qrisp_optimization.utils import QuantumBinary

## 1. Instantiate the problem

MaxCut asks for a partition of a graph's nodes into two sets such that the total weight of
the edges crossing between them is maximal. Since all solvers *minimize*,
[`MaxCutProblem`][qrisp_optimization.problems.MaxCutProblem] reports the **negative** cut
weight as its cost.

In [2]:
G = nx.Graph()
G.add_weighted_edges_from(
    [(0, 1, 1.0), (0, 2, 2.0), (1, 2, 1.0), (1, 3, 3.0), (2, 4, 1.0), (3, 4, 2.0), (4, 5, 1.0)]
)

problem = MaxCutProblem(G)
print(f"{problem.n} nodes, {len(problem.edges)} edges")
print("QAOA-solvable:      ", isinstance(problem, QAOARequirements))
print("Bruteforce-solvable:", isinstance(problem, BruteforceRequirements))

6 nodes, 7 edges
QAOA-solvable:       True
Bruteforce-solvable: True


A solution is a binary vector `x`, where `x[i]` is the side of node `problem.nodes[i]`.
You can evaluate any candidate classically:

In [3]:
x = np.array([0, 1, 1, 0, 0, 1])
problem.cl_cost_function(x)

-8.0

## 2. Get a reference solution with `Bruteforce`

For small instances, exhaustive search gives you every optimal solution. This is useful to
judge how well a quantum algorithm performs. It scales as $2^n$, so only use it on toy
problems.

In [4]:
optimal = Bruteforce(problem).solve()
optimal_cost = problem.cl_cost_function(optimal[0])

print(f"optimal cost: {optimal_cost}")
for x in optimal:
    print(x)

optimal cost: -10.0
[0 0 1 1 0 1]
[1 1 0 0 1 0]


## 3. Solve with `QAOA`

[`QAOA`][qrisp_optimization.QAOA] takes the problem and a **factory** for the quantum
argument. A factory (rather than a ready-made `QuantumArray`) is needed because every
call to [`ansatz`][qrisp_optimization.qaoa.QAOA.ansatz] builds its circuit on a fresh
quantum argument.

We use one [`QuantumBinary`][qrisp_optimization.utils.QuantumBinary] per node, so
measurement outcomes are `0`/`1` vectors that `cl_cost_function` understands directly.

In [5]:
qaoa = QAOA(
    problem=problem,
    qarg=lambda: QuantumArray(qtype=QuantumBinary(), shape=problem.n),
)

The variational angles `theta` have length `2 * p` for depth `p`: the first `p` entries are
the mixer angles $\beta$, the last `p` the cost angles $\gamma$.
[`optimize`][qrisp_optimization.qaoa.QAOA.optimize] minimizes the expected cost over these
angles with SciPy and returns the optimized angles.

In [6]:
depth = 3
theta0 = np.full(2 * depth, 0.5)  # could also be random or a warm start

theta = qaoa.optimize(theta0, max_iter=100)
theta

array([1.1568356 , 1.76660757, 1.28912876, 0.43375812, 1.66455094,
       0.15450863])

To get the solution, build the ansatz with the optimized angles and measure it.

In [7]:
optimal_set = {tuple(x) for x in optimal}


def report(meas: dict, top: int = 5) -> None:
    # print the most likely outcomes and the total probability of measuring an optimum
    p_opt = sum(p for x, p in meas.items() if tuple(x) in optimal_set)
    p_random = len(optimal_set) / 2**problem.n
    print(f"P(optimal) = {p_opt:.3f}   (random guessing: {p_random:.3f})")
    for x, p in sorted(meas.items(), key=lambda kv: -kv[1])[:top]:
        marker = "  <- optimal" if tuple(x) in optimal_set else ""
        print(f"  {x}  cost={problem.cl_cost_function(np.array(x)):6.1f}  p={p:.3f}{marker}")


meas = qaoa.ansatz(theta).get_measurement()
report(meas)

P(optimal) = 0.227   (random guessing: 0.031)
  [0 0 1 1 0 1]  cost= -10.0  p=0.113  <- optimal
  [1 1 0 0 1 0]  cost= -10.0  p=0.113  <- optimal
  [0 1 1 0 1 0]  cost=  -9.0  p=0.073
  [1 0 0 1 0 1]  cost=  -9.0  p=0.073
  [0 1 1 0 0 1]  cost=  -8.0  p=0.053


### Choosing a different metric

By default `optimize` minimizes [`exp_val`][qrisp_optimization.metrics.exp_val], the
expectation value of the cost. Any callable with signature `metric(qarg, problem) -> float`
can be used instead.

A popular alternative is the *conditional value at risk* (CVaR): only the best
`alpha` fraction of the outcomes is averaged, which rewards concentrating probability on
good solutions rather than improving the average.

In [8]:
def cvar(alpha: float):
    def metric(qarg, problem) -> float:
        meas = qarg.get_measurement()
        costs = sorted((problem.cl_cost_function(np.array(x)), p) for x, p in meas.items())
        total, remaining = 0.0, alpha
        for cost, p in costs:
            take = min(p, remaining)
            total += take * cost
            remaining -= take
            if remaining <= 0:
                break
        return total / alpha

    return metric


theta_cvar = qaoa.optimize(theta0, metric=cvar(0.2), max_iter=100)
report(qaoa.ansatz(theta_cvar).get_measurement())

P(optimal) = 0.249   (random guessing: 0.031)
  [0 0 1 1 0 1]  cost= -10.0  p=0.125  <- optimal
  [1 1 0 0 1 0]  cost= -10.0  p=0.125  <- optimal
  [0 0 1 1 1 0]  cost=  -7.0  p=0.066
  [1 1 0 0 0 1]  cost=  -7.0  p=0.066
  [0 1 1 0 0 1]  cost=  -8.0  p=0.060


`optimize` also forwards `method` and `options` to
[`scipy.optimize.minimize`][scipy.optimize.minimize], e.g.
`qaoa.optimize(theta0, method="Nelder-Mead", options={"xatol": 1e-3})`.

## 4. Solve with `LRQAOA`

[`LRQAOA`][qrisp_optimization.LRQAOA] needs exactly the same requirements as `QAOA`, so the
same problem works unchanged. Instead of optimizing the angles, LR-QAOA ramps them linearly
(an adiabatic-like schedule), so there is no classical optimization loop. You only pick the
depth and the two ramp slopes.

In [9]:
lr_qaoa = LRQAOA(
    problem=problem,
    qarg=lambda: QuantumArray(qtype=QuantumBinary(), shape=problem.n),
)

meas = lr_qaoa.ansatz(depth=10, d_gamma=0.6, d_beta=0.6).get_measurement()
report(meas)

P(optimal) = 0.465   (random guessing: 0.031)
  [0 0 1 1 0 1]  cost= -10.0  p=0.233  <- optimal
  [1 1 0 0 1 0]  cost= -10.0  p=0.233  <- optimal
  [0 1 1 0 1 0]  cost=  -9.0  p=0.076
  [1 0 0 1 0 1]  cost=  -9.0  p=0.076
  [0 1 0 0 1 0]  cost=  -9.0  p=0.076


## 5. Swap the problem

Because the solvers only know the requirements interface, solving a different problem means
changing only the line that constructs it. Here is a small QUBO, $\min_x x^T Q x$:

In [10]:
Q = np.array(
    [
        [1.0, -3.0, -2.0, 2.0],
        [0.0, 2.5, -2.0, 0.0],
        [0.0, 0.0, 3.0, -2.5],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
problem = QUBOProblem(Q)  # the only line that changed

optimal = Bruteforce(problem).solve()
optimal_set = {tuple(x) for x in optimal}

qaoa = QAOA(problem=problem, qarg=lambda: QuantumArray(qtype=QuantumBinary(), shape=problem.n))
theta = qaoa.optimize(np.full(2 * depth, 0.5))
report(qaoa.ansatz(theta).get_measurement())

P(optimal) = 0.197   (random guessing: 0.062)
  [0 0 0 0]  cost=   0.0  p=0.205
  [1 1 1 0]  cost=  -0.5  p=0.197  <- optimal
  [1 1 1 1]  cost=   0.0  p=0.151
  [0 0 1 1]  cost=   1.5  p=0.111
  [0 0 0 1]  cost=   1.0  p=0.096


## Next steps

- Your problem is not implemented yet? See [Add a new problem](new-problem.ipynb).
- You want to try a different algorithm? See [Add a new algorithm](new-algorithm.ipynb).